# Task A — Run 25 blend: validation evaluation and released-test predictions

This notebook uses the Run 25 three-member blend with the weights fixed before this run:

- character-only TF-IDF/SVM: **0.60**
- demojized MuRIL with two reinitialized layers: **0.33**
- word+character TF-IDF/SVM: **0.07**

First, all three models are fitted only on the original Task A training file and predict the released labelled validation file. That prediction is evaluated with macro-F1, accuracy and a classification report. The validation labels are not used during this stage.

Second, the models are refitted on the original training rows plus the released labelled validation rows, then predict the separate released test file. The test output is a plain `predictions.csv`; no CodaBench ZIP is created. Expected runtime is roughly 1–2 hours on a Kaggle T4 ×2.

In [ ]:
import os, pathlib, shutil, subprocess, sys, json
WORK = '/kaggle/working/hastika'
if os.path.isdir(WORK + '/.git'):
    subprocess.run(['git', '-C', WORK, 'pull', '--ff-only', 'origin', 'task-b'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', 'task-b', '--depth', '1', 'https://github.com/robinpnalex/Hastika-ICON2026.git', WORK], check=True)
os.chdir(WORK)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')
sys.path.insert(0, os.path.join(WORK, 'src'))
pathlib.Path('artifacts/logs').mkdir(parents=True, exist_ok=True)
from urllib.request import urlretrieve
raw_base = 'https://raw.githubusercontent.com/robinpnalex/Hastika-ICON2026/task-b/'
for rel in ['data/raw/hastika_binary_validation.csv', 'data/raw/hastika_binary_test.csv']:
    p = pathlib.Path(rel)
    if not p.exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(raw_base + rel, p)
    assert p.exists() and p.stat().st_size > 0, f'missing required data: {rel}'
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
import numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from hastika.common.preprocessing import clean, dedupe_index
assert torch.cuda.is_available(), 'enable a GPU'
print('gpu:', torch.cuda.get_device_name(0))

W_CHAR = 0.60
W_MURIL = 0.33
W_WORDCHAR = 0.07
assert abs(W_CHAR + W_MURIL + W_WORDCHAR - 1.0) < 1e-9

def run(cmd, log=None):
    print('$', ' '.join(map(str, cmd)), flush=True)
    fh = open(log, 'w') if log else None
    p = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f'exit {p.returncode}: {cmd}')


## 1. Fit on original training data and evaluate released validation

The canonical validation-input file and labelled validation file are joined by `id` to guarantee that probabilities and labels are aligned. No validation label is supplied to either training command.

In [ ]:
from pathlib import Path
train_path = Path('data/raw/binary_train.csv')
val_inputs_path = Path('data/raw/binary_validation_inputs.csv')
val_labels_path = Path('data/raw/hastika_binary_validation.csv')
train = pd.read_csv(train_path)
val_inputs = pd.read_csv(val_inputs_path)
val_labels = pd.read_csv(val_labels_path)
val = val_inputs[['id']].merge(val_labels, on='id', how='left', validate='one_to_one')
assert val['Label'].notna().all() and val['id'].is_unique and len(val) == 806
VAL_CHAR = 'a_run27_val_char_only'
VAL_WORDCHAR = 'a_run27_val_wordchar'
VAL_MURIL = 'a_run27_val_muril_reinit2'
run([sys.executable, '-u', '-m', 'hastika.models.baseline_svm', '--task', 'a', '--tag', VAL_CHAR, '--demojize', '--full-fit'], log=f'artifacts/logs/{VAL_CHAR}.log')
run([sys.executable, '-u', '-m', 'hastika.models.baseline_svm', '--task', 'a', '--tag', VAL_WORDCHAR, '--demojize', '--full-fit'], log=f'artifacts/logs/{VAL_WORDCHAR}.log')
run([sys.executable, '-u', '-m', 'hastika.models.muril', '--tag', VAL_MURIL, '--model', 'google/muril-base-cased', '--folds', '1', '--seeds', '42', '--epochs', '6', '--bs', '8', '--grad-accum', '2', '--eval-bs', '32', '--select', 'last', '--reinit-layers', '2'], log=f'artifacts/logs/{VAL_MURIL}.log')
char_p = np.load(Path('artifacts/runs') / VAL_CHAR / 'test_probs.npy')
wordchar_p = np.load(Path('artifacts/runs') / VAL_WORDCHAR / 'test_probs.npy')
muril_p = np.load(Path('artifacts/runs') / VAL_MURIL / 'test_probs.npy')
assert char_p.shape == wordchar_p.shape == muril_p.shape == (806, 2)
blend_p = W_CHAR * char_p[:, 1] + W_MURIL * muril_p[:, 1] + W_WORDCHAR * wordchar_p[:, 1]
y_val = (val['Label'] == 'Hate').astype(int).to_numpy()
pred_val = (blend_p > 0.5).astype(int)
val_summary = {'macro_f1': f1_score(y_val, pred_val, average='macro'), 'accuracy': accuracy_score(y_val, pred_val), 'rows': len(y_val), 'w_char': W_CHAR, 'w_muril': W_MURIL, 'w_wordchar': W_WORDCHAR}
print('VALIDATION RESULTS:', val_summary)
print(classification_report(y_val, pred_val, target_names=['Non-Hate', 'Hate'], digits=4, zero_division=0))
VAL_OUT = Path('/kaggle/working/task_a_run27_outputs')
VAL_OUT.mkdir(parents=True, exist_ok=True)
pd.DataFrame({'id': val['id'], 'label': np.where(pred_val == 1, 'Hate', 'Non-Hate')}).to_csv(VAL_OUT / 'validation_predictions.csv', index=False)
json.dump(val_summary, open(VAL_OUT / 'validation_summary.json', 'w'), indent=2)


## 2. Refit on train plus validation and predict the released test set

After the validation measurement is saved, the labelled validation rows are allowed into the final fit. The resulting predictions are for `hastika_binary_test.csv`, not the already-scored validation file.

In [ ]:
test_path = Path('data/raw/hastika_binary_test.csv')
test_df = pd.read_csv(test_path)
combined = pd.concat([train, val_labels], ignore_index=True).drop_duplicates('id', keep='first').reset_index(drop=True)
assert len(combined) == 7252 and combined['id'].is_unique
combined_path = Path('data/derived/task_a_run27_train_plus_validation.csv')
combined_path.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(combined_path, index=False)
CANON_TRAIN = Path('data/raw/binary_train.csv')
CANON_TEST = Path('data/raw/binary_validation_inputs.csv')
BACKUP_TRAIN = Path('data/raw/binary_train.csv.before_run27')
BACKUP_TEST = Path('data/raw/binary_validation_inputs.csv.before_run27')
assert not BACKUP_TRAIN.exists() and not BACKUP_TEST.exists(), 'stale Run 27 backup exists; inspect before rerunning'
shutil.copy2(CANON_TRAIN, BACKUP_TRAIN)
shutil.copy2(CANON_TEST, BACKUP_TEST)
shutil.copy2(combined_path, CANON_TRAIN)
shutil.copy2(test_path, CANON_TEST)
TEST_CHAR = 'a_run27_test_char_only'
TEST_WORDCHAR = 'a_run27_test_wordchar'
TEST_MURIL = 'a_run27_test_muril_reinit2'
try:
    run([sys.executable, '-u', '-m', 'hastika.models.baseline_svm', '--task', 'a', '--tag', TEST_CHAR, '--demojize', '--full-fit'], log=f'artifacts/logs/{TEST_CHAR}.log')
    run([sys.executable, '-u', '-m', 'hastika.models.baseline_svm', '--task', 'a', '--tag', TEST_WORDCHAR, '--demojize', '--full-fit'], log=f'artifacts/logs/{TEST_WORDCHAR}.log')
    run([sys.executable, '-u', '-m', 'hastika.models.muril', '--tag', TEST_MURIL, '--model', 'google/muril-base-cased', '--folds', '1', '--seeds', '42', '--epochs', '6', '--bs', '8', '--grad-accum', '2', '--eval-bs', '32', '--select', 'last', '--reinit-layers', '2'], log=f'artifacts/logs/{TEST_MURIL}.log')
    char_test = np.load(Path('artifacts/runs') / TEST_CHAR / 'test_probs.npy')[:, 1]
    wordchar_test = np.load(Path('artifacts/runs') / TEST_WORDCHAR / 'test_probs.npy')[:, 1]
    muril_test = np.load(Path('artifacts/runs') / TEST_MURIL / 'test_probs.npy')[:, 1]
    assert len(char_test) == len(wordchar_test) == len(muril_test) == len(test_df) == 806
    test_blend = W_CHAR * char_test + W_MURIL * muril_test + W_WORDCHAR * wordchar_test
    test_pred = np.where(test_blend > 0.5, 'Hate', 'Non-Hate')
    test_out = VAL_OUT / 'test_predictions.csv'
    pd.DataFrame({'id': test_df['id'], 'label': test_pred}).to_csv(test_out, index=False)
    json.dump({'rows': len(test_pred), 'w_char': W_CHAR, 'w_muril': W_MURIL, 'w_wordchar': W_WORDCHAR, 'threshold': 0.5}, open(VAL_OUT / 'test_config.json', 'w'), indent=2)
    print('TEST PREDICTIONS:', test_out)
    print('test class counts:', pd.Series(test_pred).value_counts().to_dict())
finally:
    shutil.move(str(BACKUP_TRAIN), str(CANON_TRAIN))
    shutil.move(str(BACKUP_TEST), str(CANON_TEST))
    print('restored canonical repository data files')


## Outputs

Download `/kaggle/working/task_a_run27_outputs/validation_predictions.csv` and `validation_summary.json` for the measured validation result. Download `test_predictions.csv` for the separate released-test predictions. The validation score is meaningful because the validation labels were not used in the first-stage fit; the test predictions use all available labelled rows.